# 📤 데이터 업로드

샘플 데이터를 Azure AI Search 인덱스에 업로드합니다.

## 📋 업로드 과정

1. **CSV 파일 읽기**: pandas로 sample_data.csv 로드
2. **문서 변환**: 각 행을 검색 문서 형식으로 변환
3. **배치 업로드**: 100개씩 묶어서 업로드
4. **결과 확인**: 성공/실패 건수 출력

## 1️⃣ 라이브러리 임포트

In [1]:
import pandas as pd
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
import os

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


## 2️⃣ 환경 변수 로드

In [2]:
# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 접속 정보
endpoint = os.getenv("SEARCH_ENDPOINT")
key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

if not endpoint or not key or not index_name:
    print("❌ 환경 변수가 설정되지 않았습니다. .env 파일을 확인하세요.")
else:
    print("✅ 환경 변수 로드 완료")
    print(f"   Endpoint: {endpoint}")
    print(f"   Index Name: {index_name}")

✅ 환경 변수 로드 완료
   Endpoint: https://ai-search-foundry-iq-cj.search.windows.net
   Index Name: products-index


## 3️⃣ CSV 데이터 읽기

sample_data.csv 파일을 읽어옵니다.

In [3]:
# CSV 파일 경로
csv_file = "../00-data/sample_data.csv"

# CSV 읽기
df = pd.read_csv(csv_file)

print(f"✅ CSV 파일 로드 완료")
print(f"   총 데이터 수: {len(df)}개")
print(f"   컬럼: {list(df.columns)}")

# 상위 3개 데이터 미리보기
print("\n📊 데이터 미리보기:")
df.head(3)

✅ CSV 파일 로드 완료
   총 데이터 수: 247개
   컬럼: ['id', 'title', 'brand', 'category', 'normal_price', 'image_link', 'review']

📊 데이터 미리보기:


,id,title,brand,category,normal_price,image_link,review
0,1,압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물),압소바,유아동,39000,https://foundryiq-image-gallery.azurewebsites....,신생아 백일 선물로 압소바6 레코딸랑이세트를 구매했는데 디자인이 귀여워서 선물 받는...
1,2,[압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1),압소바,유아동,40000,https://foundryiq-image-gallery.azurewebsites....,친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부...
2,3,블루독베이비[hu] WH)블루독남아손수건SET 41170-006-01 손수건/타올/...,블루독베이비,유아동,22190,https://foundryiq-image-gallery.azurewebsites....,출산 준비하면서 블루독베이비 손수건 세트를 구입했는데 10장이나 들어 있어서 정말 ...


## 4️⃣ 검색 문서 준비

CSV 데이터를 Azure AI Search 문서 형식으로 변환합니다.

### 필드 매핑
- `id`: String으로 변환 (필수 key 필드)
- `title`: 상품명
- `brand`: 브랜드명
- `category`: 카테고리
- `normal_price`: 가격 (Int32)
- `review`: 제품에 대한 최근 리뷰

In [4]:
# 검색 문서 준비
documents = []

for _, row in df.iterrows():
    doc = {
        "id": str(row['id']),  # String 타입으로 변환
        "title": row['title'],
        "brand": row['brand'],
        "category": row['category'],
        "normal_price": int(row['normal_price']),
        "review": row['review'],  # 사용후기 텍스트
    }
    documents.append(doc)

print(f"✅ {len(documents)}개 문서 준비 완료")

# 첫 번째 문서 예시
print("\n📄 문서 예시:")
print(documents[0])

✅ 247개 문서 준비 완료

📄 문서 예시:
{'id': '1', 'title': '압소바6 (ATA367P2M13) 레코딸랑이세트(신생아 백일 선물)', 'brand': '압소바', 'category': '유아동', 'normal_price': 39000, 'review': '신생아 백일 선물로 압소바6 레코딸랑이세트를 구매했는데 디자인이 귀여워서 선물 받는 가족도 매우 좋아했어요. 재질이 부드러워 아기 피부에 자극 없이 안전하게 사용할 수 있었고, 장난감 소리도 은은해 아이가 잘 집중하더라고요. 크기도 아기 손에 딱 맞아 잡기 편하고 세트 구성이 실용적이라 만족스러웠습니다. 가격대도 적당해 부담 없이 선물하기 좋은 제품입니다.'}


## 5️⃣ 인덱스에 업로드

준비한 문서를 Azure AI Search 인덱스에 배치 업로드합니다.

### 배치 업로드
- 한 번에 최대 1000개까지 업로드 가능
- 이 실습에서는 100개씩 나누어 업로드

In [5]:
# SearchClient 초기화
credential = AzureKeyCredential(key)
client = SearchClient(endpoint=endpoint, index_name=index_name, credential=credential)

print(f"📤 Index '{index_name}'에 업로드 중...\n")

# 배치 업로드
batch_size = 100
total = len(documents)
total_succeeded = 0
total_failed = 0

for i in range(0, total, batch_size):
    batch = documents[i:i+batch_size]
    
    try:
        result = client.upload_documents(documents=batch)
        
        succeeded = len([r for r in result if r.succeeded])
        failed = len([r for r in result if not r.succeeded])
        
        total_succeeded += succeeded
        total_failed += failed
        
        print(f"📤 [{i+succeeded}/{total}] 업로드 완료 (성공: {succeeded}, 실패: {failed})")
        
        # 실패한 문서 상세 정보
        if failed > 0:
            for r in result:
                if not r.succeeded:
                    print(f"   ⚠️  실패: {r.key} - {r.error_message}")
    
    except Exception as e:
        print(f"❌ 배치 업로드 실패: {str(e)}")
        total_failed += len(batch)

print("\n" + "=" * 60)
print(f"✅ 업로드 완료!")
print(f"   총 문서: {total}개")
print(f"   성공: {total_succeeded}개")
print(f"   실패: {total_failed}개")
print("=" * 60)

📤 Index 'products-index'에 업로드 중...

📤 [100/247] 업로드 완료 (성공: 100, 실패: 0)
📤 [200/247] 업로드 완료 (성공: 100, 실패: 0)
📤 [247/247] 업로드 완료 (성공: 47, 실패: 0)

✅ 업로드 완료!
   총 문서: 247개
   성공: 247개
   실패: 0개


## 6️⃣ 업로드 검증

인덱스의 문서 개수를 확인하여 업로드가 정상적으로 완료되었는지 검증합니다.

In [8]:
# 간단한 검색으로 문서 개수 확인
# Azure AI Search는 업로드 후 인덱싱이 비동기로 진행되므로
# 즉시 count를 조회하면 0이 나올 수 있습니다. 잠시 기다린 뒤 재시도합니다.
import time

doc_count = 0
for attempt in range(10):
    try:
        results = client.search(
            search_text="*",
            top=0,
            include_total_count=True
        )
        doc_count = results.get_count()
        if doc_count == len(documents):
            break
    except Exception as e:
        print(f"⏳ 재시도 {attempt + 1}/10... ({e})")
    time.sleep(2)

print(f"\n📊 인덱스 '{index_name}' 문서 개수: {doc_count}개")

if doc_count == len(documents):
    print("✅ 모든 문서가 정상적으로 업로드되었습니다!")
else:
    print(f"⚠️  예상: {len(documents)}개, 실제: {doc_count}개 (인덱싱이 아직 진행 중일 수 있습니다)")



📊 인덱스 'products-index' 문서 개수: 247개
✅ 모든 문서가 정상적으로 업로드되었습니다!


## 7️⃣ 샘플 검색 테스트

업로드된 데이터로 간단한 검색을 테스트합니다.

In [9]:
# 간단한 검색 테스트
test_query = "압소바"

results = client.search(
    search_text=test_query,
    top=3,
    include_total_count=True
)

print(f"\n🔍 테스트 검색어: '{test_query}'")
print(f"📊 총 {results.get_count()}개 결과 (상위 3개 표시)\n")
print("=" * 80)

print("=" * 80)

for idx, result in enumerate(results, 1):
    print(f"{idx}. {result['title']}")
    print(f"   브랜드: {result['brand']} | 카테고리: {result['category']} | 가격: {result['normal_price']:,}원")
    print(f"   점수: {result['@search.score']:.2f}")
    print(f"   리뷰: {result['review']}")
    print(f"   Origin: {result}")
    print()
    


🔍 테스트 검색어: '압소바'
📊 총 3개 결과 (상위 3개 표시)

1. [압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)
   브랜드: 압소바 | 카테고리: 유아동 | 가격: 40,000원
   점수: 16.95
   리뷰: 친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부에 자극 없고, 딸랑이 소리도 은은해서 너무 마음에 들었습니다. 선물포장도 깔끔하게 되어 있어 바로 전달하기 좋았고, 디자인도 귀여워서 받는 분이 정말 좋아했어요. 실용적이면서도 예쁜 선물로 강력 추천합니다.
   Origin: {'normal_price': 40000, 'brand': '압소바', 'review': '친구 아기 출산선물로 압소바 티노딸랑이세트를 구매했어요. 부드러운 소재라 아기 피부에 자극 없고, 딸랑이 소리도 은은해서 너무 마음에 들었습니다. 선물포장도 깔끔하게 되어 있어 바로 전달하기 좋았고, 디자인도 귀여워서 받는 분이 정말 좋아했어요. 실용적이면서도 예쁜 선물로 강력 추천합니다.', 'id': '2', 'category': '유아동', 'title': '[압소바] 출산선물 티노딸랑이세트 (선물포장) (ATA367P1)', '@search.score': 16.953777, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}

2. [압소바1] 곰고무연사목욕타올+손수건+인형세트 MIAZA-10911 [임신/출산선물]
   브랜드: 압소바 | 카테고리: 유아동 | 가격: 96,000원
   점수: 11.71
   리뷰: 신생아 목욕용으로 압소바 곰고무연사목욕타올 세트를 구입했는데 부드러운 소재 덕분에 아기의 피부에 자극 없이 잘 사용하고 있어요. 주머니가 있어 손수건과 귀여운 곰 인형도 함께 있어 목욕 시간이 더 즐거워졌습니다. 세트 구성도 알차고 세탁 후에도 변형 없이 오래 쓸 수 있어 만족스럽습니다. 임신·출산 선

---

## 🎯 다음 단계

데이터 업로드가 완료되었습니다! 이제 다음 단계로 진행하세요:

1. **`03-keyword_search.ipynb`** - 다양한 키워드 검색 실습
2. **Azure Portal** - 검색 탐색기로 결과 확인

---

## 💡 참고 사항

### 업로드 실패 시 확인 사항
1. **인덱스 생성 확인**: `01-create_index.ipynb`를 먼저 실행했는지 확인
2. **필드 타입 확인**: CSV의 데이터 타입과 인덱스 스키마가 일치하는지 확인
3. **ID 중복 확인**: 같은 ID로 재업로드 시 문서가 덮어씌워짐

### 배치 업로드 팁
- 한 번에 최대 1000개까지 업로드 가능
- 대용량 데이터는 100~1000개 단위로 나누어 업로드 권장
- 실패한 문서는 개별적으로 재시도 가능

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 02-1: 인덱스 생성](01-create_index.ipynb) | [워크숍 홈](../README.md) | [Lab 02-3: 키워드 검색](03-keyword_search.ipynb) |